# FILE 2/3 — Luồng B, Chặng 2: Fine-tune B-CanTho → HCMC (seed = 1407)

Notebook nay la **buoc thu hai** cua lo trinh tuan tu B. Khoi tao tu model B-CanTho da
sinh ra o FILE 1, fine-tune tiep tren HCMC de tao ra **model B cuoi cung** (seed 1407).

**Vi tri trong toan bo quy trinh:**
- FILE 1: COCO → CanTho → model B-CanTho (da xong)
- **FILE 2 (notebook nay):** B-CanTho → HCMC → model B cuoi cung (seed 1407)
- FILE 3: train luồng A + danh gia ca A va B tren 3 tap test + bang so sanh

**BAT BUOC chay FILE 1 xong truoc**, vi FILE 2 can model B-CanTho lam diem khoi tao.
Neu chua co model B-CanTho, cell Buoc 3 se bao loi khong tim thay file.

**Luu y:** seed=1407, giu nguyen cach chia du lieu, dung cau hinh Bang 3.4.

## Buoc 1: Cai dat thu vien

In [ ]:
!pip install ultralytics -q
print("Da cai dat xong ultralytics")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.4/46.4 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 30.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.9/68.9 kB 6.9 MB/s eta 0:00:00
Da cai dat xong ultralytics


## Buoc 2: Ket noi Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


## Buoc 3: Khai bao duong dan va cau hinh

In [ ]:
import os

SEED = 1407

# ===== MODEL KHOI TAO =====
# Model B-CanTho sinh ra tu FILE 1 (COCO->CanTho seed 1407)
PRETRAINED_B_CANTHO = "/content/drive/MyDrive/Do an AI giao thong/Baseline/Ketqua_Baseline/finetune_cantho_seed1407/weights/best.pt"

# ===== DATASET (goc tren Drive) =====
DRIVE_DIR_HCMC = "/content/drive/MyDrive/Do an AI giao thong/Dataset/Vehicle Counting AIC HCMC 2020.v1i.yolov8_701515"

# ===== DATASET (copy ve local) =====
LOCAL_DIR_HCMC = "/content/hcmc_701515"

# ===== THU MUC LUU KET QUA =====
PROJECT_DIR = "/content/drive/MyDrive/Do an AI giao thong/Baseline/Ketqua_Baseline"
NAME_B_FINAL = "finetune_hcmc_from_cantho_seed1407"   # model B cuoi cung (seed moi)

# ===== CAU HINH HUAN LUYEN (dung Bang 3.4) =====
IMGSZ = 640
BATCH = 16
EPOCHS = 100
PATIENCE = 20
LR0 = 0.001

print("Kiem tra su ton tai:")
print("  Model B-CanTho (tu FILE 1):", os.path.exists(PRETRAINED_B_CANTHO))
print("  Dataset HCMC (Drive)       :", os.path.exists(DRIVE_DIR_HCMC))
if not os.path.exists(PRETRAINED_B_CANTHO):
    print("\n[LOI] Chua tim thay model B-CanTho. Hay chay xong FILE 1 truoc!")

Kiem tra su ton tai:
  Model B-CanTho (tu FILE 1): True
  Dataset HCMC (Drive)       : True


## Buoc 4: Copy dataset HCMC ve o cung local

Neu gap loi "Transport endpoint is not connected", chi can chay lai dung cell nay.

In [ ]:
import shutil, time

if os.path.exists(LOCAL_DIR_HCMC):
    shutil.rmtree(LOCAL_DIR_HCMC)
t0 = time.time()
try:
    shutil.copytree(DRIVE_DIR_HCMC, LOCAL_DIR_HCMC)
    print(f"[OK] Da copy HCMC ve {LOCAL_DIR_HCMC} ({time.time()-t0:.1f}s)")
except shutil.Error as e:
    print(f"[CANH BAO] {len(e.args[0])} file loi tam thoi khi copy, kiem tra lai so luong sau.")

DATA_YAML_HCMC_LOCAL = f"{LOCAL_DIR_HCMC}/data.yaml"
print("Yaml HCMC (local) ton tai:", os.path.exists(DATA_YAML_HCMC_LOCAL))

[OK] Da copy HCMC ve /content/hcmc_701515 (844.6s)
Yaml HCMC (local) ton tai: True


## Buoc 5: Huan luyen chang 2 — B-CanTho → HCMC (seed 1407)

Khoi tao tu model B-CanTho (KHONG phai tu COCO goc), fine-tune tiep tren HCMC. Day chinh
la diem tao nen tinh "tuan tu" cua lo trinh B: chang cuoi xuat phat tu model da hoc CanTho.
Buoc nay co huan luyen that, mat khoang 1-1.5 gio.

In [5]:
from ultralytics import YOLO

model_b = YOLO(PRETRAINED_B_CANTHO)   # khoi tao tu model B-CanTho (da hoc CanTho)

results = model_b.train(
    data=DATA_YAML_HCMC_LOCAL,
    epochs=EPOCHS,
    patience=PATIENCE,
    imgsz=IMGSZ,
    batch=BATCH,
    lr0=LR0,
    seed=SEED,
    project=PROJECT_DIR,
    name=NAME_B_FINAL,
    exist_ok=True,
    verbose=True,
)
print("\nDa huan luyen xong chang 2 (B-CanTho -> HCMC, seed 1407).")
BEST_B_FINAL = os.path.join(PROJECT_DIR, NAME_B_FINAL, "weights", "best.pt")
print("Model B cuoi cung luu tai:", BEST_B_FINAL)
print("Ton tai:", os.path.exists(BEST_B_FINAL))
print("\n>>> DA XONG FILE 2. Tiep theo chay FILE 3 (train luong A + danh gia + so sanh).")

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.
Ultralytics 8.4.143 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/hcmc_701515/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscri